# FM-PCC Colab Workflow

---

## Run Order
1. Mount Drive
2. Clone/Update FM-PCC
3. Boost startup cache wiring
4. Install Miniconda + Create `FMPCC` env
5. Install D3IL
6. Install requirements with pinned compatibility
7. Set runtime env variables
8. Optional W&B relogin
9. Verify dependencies
10. Prepare dataset
11. Smoke test
12. Full train
13. Eval + visualization
14. Archive logs


## Remember to clear cell outputs before run!



In [1]:
# @title Notebook Version Marker
from datetime import datetime
import pytz

# Change 'UTC' to your local timezone if preferred (e.g., 'US/Eastern', 'Asia/Shanghai')
TIMEZONE = 'UTC'

now = datetime.now(pytz.timezone(TIMEZONE))
version_mark = now.strftime("%Y.%m.%d_%H%M%S")

print("=" * 40)
print(f"SESSION VERSION: {version_mark}")
print(f"START TIME     : {now.strftime('%A, %b %d, %Y - %I:%M:%S %p %Z')}")
print("=" * 40)
print("FMv3 -> aw = 10 ODE 20 euler, ie DPCC replicate), continue train 90 + Eval")

SESSION VERSION: 2026.04.23_214102
START TIME     : Thursday, Apr 23, 2026 - 09:41:02 PM UTC
FMv3 -> aw = 10 ODE 20 euler, ie DPCC replicate), continue train 90 + Eval


## 1) Mount Google Drive



In [2]:
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

FMPCC_ROOT = '/content/drive/MyDrive/FMPCC'
REPO = f'{FMPCC_ROOT}/FM-PCC'
os.makedirs(FMPCC_ROOT, exist_ok=True)

print('Drive mounted')
print('Repo path:', REPO)



Mounted at /content/drive
Drive mounted
Repo path: /content/drive/MyDrive/FMPCC/FM-PCC


## 2) Clone or Update FM-PCC



here will encounter some problem when first time git pull, pending debug

In [ ]:
# @title Git Repository Sync
# @markdown Set this to 1 to replace any edited GitHub files with the latest versions.
# @markdown Your new files/notes will NOT be deleted.
OVERWRITE_LOCAL_CHANGES = "1" # @param [0, 1]
UPDATE_REPO = 1 # @param [0, 1]

import os
os.environ['OVERWRITE_LOCAL_CHANGES'] = str(OVERWRITE_LOCAL_CHANGES)
os.environ['UPDATE_REPO'] = str(UPDATE_REPO)

In [ ]:
%%bash
set -e

ROOT="/content/drive/MyDrive/FMPCC"
REPO="$ROOT/FM-PCC"
BRANCH="main" # You can change this if you ever decide to switch branches

mkdir -p "$ROOT"
cd "$ROOT"

if [ ! -d "$REPO/.git" ]; then
  echo "Cloning fresh repository..."
  git clone --recurse-submodules https://github.com/ghubliming/FM-PCC.git
else
  cd "$REPO"

  if [ "$UPDATE_REPO" != "1" ]; then
    echo "Repo update disabled. Using existing local checkout."
  else
    echo "Fetching latest updates from GitHub..."
    git fetch origin "$BRANCH"

    if [ "$OVERWRITE_LOCAL_CHANGES" = "1" ]; then
      echo "OVERWRITE_LOCAL_CHANGES=1: Resetting tracked files to match remote."
      # This command ONLY affects files that exist in the Git repo.
      # It does NOT touch your new notes or results.
      git reset --hard "origin/$BRANCH"
      git submodule update --init --recursive
    else
      # Check if there are any local edits to GitHub files
      CHANGED_FILES="$(git status --porcelain | grep '^ M' || true)"
      if [ -n "$CHANGED_FILES" ]; then
        echo "WARNING: Local edits detected in GitHub files. Skipping update to protect them."
        echo "Set OVERWRITE_LOCAL_CHANGES=1 to replace these files with the remote version."
      else
        echo "No conflicts found. Updating..."
        git merge "origin/$BRANCH"
        git submodule update --init --recursive
      fi
    fi
  fi
fi

echo "------------------------------------------------"
echo "Repo ready: $REPO"
echo "Current Branch: $(git branch --show-current)"
echo "------------------------------------------------"

# This part specifically shows you what is yours (notes/new files)
echo "Your Local Notes & New Files (Not in Git):"
UNTRACKED="$(git ls-files --others --exclude-standard)"
if [ -z "$UNTRACKED" ]; then
  echo "  (None)"
else
  echo "$UNTRACKED"
fi

Fetching latest updates from GitHub...
OVERWRITE_LOCAL_CHANGES=1: Resetting tracked files to match remote.
HEAD is now at 586b4d7 Merge pull request #29 from ghubliming/update_into_FM
------------------------------------------------
Repo ready: /content/drive/MyDrive/FMPCC/FM-PCC
Current Branch: main
------------------------------------------------
Your Local Notes & New Files (Not in Git):
args_resume_1.json
args_resume_10.json
args_resume_100.json
args_resume_101.json
args_resume_102.json
args_resume_103.json
args_resume_104.json
args_resume_105.json
args_resume_106.json
args_resume_107.json
args_resume_108.json
args_resume_109.json
args_resume_11.json
args_resume_110.json
args_resume_111.json
args_resume_112.json
args_resume_113.json
args_resume_114.json
args_resume_115.json
args_resume_116.json
args_resume_117.json
args_resume_118.json
args_resume_119.json
args_resume_12.json
args_resume_120.json
args_resume_121.json
args_resume_122.json
args_resume_123.json
args_resume_124.json
ar

From https://github.com/ghubliming/FM-PCC
 * branch            main       -> FETCH_HEAD
   a349a19..586b4d7  main       -> origin/main
Updating files: 100% (2073/2073), done.


## 3) Boost Startup Cache Wiring

Keeps the original `/content/miniconda3` path logic, but maps it to Drive so restarts can reuse the same conda env and pip cache.



In [3]:
%%bash
set -e

ROOT="/content/drive/MyDrive/FMPCC"
PERSIST_CONDA="$ROOT/miniconda3"
RUNTIME_CONDA="/content/miniconda3"
PIP_CACHE="$ROOT/.pip-cache"

mkdir -p "$ROOT" "$PIP_CACHE"

if [ -L "$RUNTIME_CONDA" ]; then
  rm -f "$RUNTIME_CONDA"
elif [ -d "$RUNTIME_CONDA" ]; then
  rm -rf "$RUNTIME_CONDA"
fi

ln -s "$PERSIST_CONDA" "$RUNTIME_CONDA"

echo "Runtime conda path mapped to: $PERSIST_CONDA"
echo "Persistent pip cache path: $PIP_CACHE"



Runtime conda path mapped to: /content/drive/MyDrive/FMPCC/miniconda3
Persistent pip cache path: /content/drive/MyDrive/FMPCC/.pip-cache


## 4) Install Miniconda and Create Env

Keeps Python pinned to 3.10 for compatibility with project dependencies.



In [4]:
%%bash
set -e

ROOT="/content/drive/MyDrive/FMPCC"
PERSIST_CONDA="$ROOT/miniconda3"
RUNTIME_CONDA="/content/miniconda3"
CONDA_BIN="$PERSIST_CONDA/bin/conda"
LOCAL_FALLBACK_CONDA="/content/miniconda3_runtime"
CONDA_SNAPSHOT_DIR="$ROOT/cache"
CONDA_SNAPSHOT="$CONDA_SNAPSHOT_DIR/fmpcc_conda_env.tar.gz"
FORCE_REINSTALL_CONDA="${FORCE_REINSTALL_CONDA:-0}"
REFRESH_CONDA_SNAPSHOT="${REFRESH_CONDA_SNAPSHOT:-0}"

# Ensure the runtime path points to the persistent conda directory.
if [ -L "$RUNTIME_CONDA" ]; then
  LINK_TARGET="$(readlink -f "$RUNTIME_CONDA" || true)"
  if [ "$LINK_TARGET" != "$PERSIST_CONDA" ]; then
    rm -f "$RUNTIME_CONDA"
    ln -s "$PERSIST_CONDA" "$RUNTIME_CONDA"
  fi
elif [ ! -e "$RUNTIME_CONDA" ]; then
  ln -s "$PERSIST_CONDA" "$RUNTIME_CONDA"
fi

if [ "$FORCE_REINSTALL_CONDA" = "1" ]; then
  echo "FORCE_REINSTALL_CONDA=1 -> reinstalling Miniconda"
  rm -rf "$PERSIST_CONDA"
fi

if [ ! -d "$PERSIST_CONDA" ]; then
  echo "First run detected: no persistent conda directory yet"
fi

# 1) Check if conda exists and works.
NEED_CONDA_INSTALL=0
if [ -x "$CONDA_BIN" ]; then
  chmod +x "$PERSIST_CONDA/bin/python" "$PERSIST_CONDA/bin/conda" || true
  if ! "$CONDA_BIN" --version >/tmp/conda_check.log 2>&1; then
    if grep -qi "Permission denied" /tmp/conda_check.log; then
      echo "Drive conda is not executable (permission denied). Switching to local runtime conda."
      if [ -L "$RUNTIME_CONDA" ] || [ -e "$RUNTIME_CONDA" ]; then
        rm -rf "$RUNTIME_CONDA"
      fi
      ln -s "$LOCAL_FALLBACK_CONDA" "$RUNTIME_CONDA"
      PERSIST_CONDA="$LOCAL_FALLBACK_CONDA"
      CONDA_BIN="$PERSIST_CONDA/bin/conda"
      if [ -x "$CONDA_BIN" ]; then
        echo "Reusing existing local runtime conda."
      else
        echo "No local runtime conda found yet; it will be installed now."
        NEED_CONDA_INSTALL=1
      fi
    else
      echo "Conda exists but failed health check -> reinstalling"
      rm -rf "$PERSIST_CONDA"
      NEED_CONDA_INSTALL=1
    fi
  else
    echo "Conda exists and passed health check -> skip reinstall"
  fi
else
  NEED_CONDA_INSTALL=1
fi

# 2) If there is a problem, reinstall.
if [ "$NEED_CONDA_INSTALL" = "1" ]; then
  if [ -f "$CONDA_SNAPSHOT" ]; then
    echo "Restoring conda snapshot: $CONDA_SNAPSHOT"
    rm -rf "$PERSIST_CONDA"
    mkdir -p "$PERSIST_CONDA"
    if ! tar -xzf "$CONDA_SNAPSHOT" -C "$PERSIST_CONDA" --strip-components=1; then
      echo "Snapshot restore failed -> will run installer fallback"
      rm -rf "$PERSIST_CONDA"
    fi
  fi

  if [ ! -x "$CONDA_BIN" ] || ! "$CONDA_BIN" --version >/dev/null 2>&1; then
    echo "Conda snapshot missing/broken -> installing Miniconda"
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
    bash /content/miniconda.sh -b -p "$PERSIST_CONDA" -u
  else
    echo "Conda restored from snapshot"
  fi
fi

"$CONDA_BIN" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
"$CONDA_BIN" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

if ! "$CONDA_BIN" env list | grep -q "^FMPCC "; then
  "$CONDA_BIN" create -n FMPCC python=3.10 -y -q
fi

if [ ! -x "$PERSIST_CONDA/envs/FMPCC/bin/python" ] || [ ! -x "$PERSIST_CONDA/envs/FMPCC/bin/pip" ]; then
  "$CONDA_BIN" remove -n FMPCC --all -y || true
  "$CONDA_BIN" create -n FMPCC python=3.10 -y -q
fi

# If env binaries on Drive are not executable, switch to local runtime conda and rebuild env there.
if ! "$PERSIST_CONDA/envs/FMPCC/bin/python" -V >/tmp/env_python_check.log 2>&1; then
  if grep -qi "Permission denied" /tmp/env_python_check.log; then
    echo "FMPCC env python is not executable on current path. Switching env to local runtime conda."
    if [ -L "$RUNTIME_CONDA" ] || [ -e "$RUNTIME_CONDA" ]; then
      rm -rf "$RUNTIME_CONDA"
    fi
    ln -s "$LOCAL_FALLBACK_CONDA" "$RUNTIME_CONDA"
    PERSIST_CONDA="$LOCAL_FALLBACK_CONDA"
    CONDA_BIN="$PERSIST_CONDA/bin/conda"

    if [ ! -x "$CONDA_BIN" ]; then
      wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
      bash /content/miniconda.sh -b -p "$PERSIST_CONDA" -u
    fi

    "$CONDA_BIN" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    "$CONDA_BIN" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    "$CONDA_BIN" remove -n FMPCC --all -y || true
    "$CONDA_BIN" create -n FMPCC python=3.10 -y -q
  else
    cat /tmp/env_python_check.log
    exit 1
  fi
fi

"$PERSIST_CONDA/envs/FMPCC/bin/python" -V
"$PERSIST_CONDA/envs/FMPCC/bin/pip" --version

# Save a conda snapshot for fast restore on future Colab runtimes.
mkdir -p "$CONDA_SNAPSHOT_DIR"
if [ ! -f "$CONDA_SNAPSHOT" ] || [ "$REFRESH_CONDA_SNAPSHOT" = "1" ]; then
  echo "Creating conda snapshot: $CONDA_SNAPSHOT"
  if ! tar -czf "$CONDA_SNAPSHOT" -C "$PERSIST_CONDA" .; then
    echo "Snapshot creation failed -> continuing without cache update"
  fi
else
  echo "Conda snapshot exists -> skipping snapshot refresh"
fi



Conda exists and passed health check -> skip reinstall
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
FMPCC env python is not executable on current path. Switching env to local runtime conda.
PREFIX=/content/miniconda3_runtime
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/miniconda3_runtime
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channe


EnvironmentLocationNotFound: Not a conda environment: /content/miniconda3_runtime/envs/FMPCC



## 5) Install D3IL (Install Once + Verify)

Uses editable installs for both D3IL core and `gym_avoiding_env`, but skips reinstall when editable links already exist.



```
%%bash
set -e

PIP="/content/miniconda3/envs/FMPCC/bin/pip"
REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
D3IL="$REPO/d3il"

if [ ! -d "$D3IL/.git" ]; then
  echo "d3il missing/incomplete -> recloning"
  rm -rf "$D3IL"
  git clone https://github.com/ALRhub/d3il.git "$D3IL"
fi

if "$PIP" freeze | grep -Fq "d3il/environments/d3il" && "$PIP" freeze | grep -Fq "d3il/envs/gym_avoiding_env"; then
  echo "D3IL editable installs already present; skipping reinstall"
else
  "$PIP" install -e "$D3IL/environments/d3il"
  "$PIP" install -e "$D3IL/environments/d3il/envs/gym_avoiding_env"
fi

echo "D3IL installed"
```


### After Gen4 Update, the D3IL is in FM-PCC Repo, just need install now

In [5]:
%%bash
set -e

PIP="/content/miniconda3/envs/FMPCC/bin/pip"
REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
D3IL="$REPO/d3il"

if [ ! -d "$D3IL" ]; then
  echo "ERROR: d3il directory not found at $D3IL"
  exit 1
fi

if "$PIP" freeze | grep -Fq "d3il/environments/d3il" && "$PIP" freeze | grep -Fq "d3il/envs/gym_avoiding_env"; then
  echo "D3IL editable installs already present; skipping reinstall"
else
  "$PIP" install -e "$D3IL/environments/d3il"
  "$PIP" install -e "$D3IL/environments/d3il/envs/gym_avoiding_env"
fi

echo "D3IL installed"

Obtaining file:///content/drive/MyDrive/FMPCC/FM-PCC/d3il/environments/d3il
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for environments.d3il.d3il_sim (pyproject.toml): started
  Building editable for environments.d3il.d3il_sim (pyproject.toml): finished with status 'done'
  Created wheel for environments.d3il.d3il_sim: filename=environments_d3il_d3il_sim-0.2-0.editable-py3-none-any.whl size=2970 sha256=24ef8bd2b1a2775a2158c0ba1f2486f461ac92ec694e3c7df8515bb7e3f18523
  Stored in directory: /tmp/pip-ephem-wheel-cache-6zx4kj

## 6) Install Requirements (Install Once + Verify)

Runs validation first and only installs when the environment is missing or inconsistent.



In [6]:
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
PIP="/content/miniconda3/envs/FMPCC/bin/pip"
PY="/content/miniconda3/envs/FMPCC/bin/python"
PIP_CACHE="/content/drive/MyDrive/FMPCC/.pip-cache"
CACHE_DIR="/content/drive/MyDrive/FMPCC/cache"
REQ_FILE="$REPO/requirements.txt"
REQ_HASH="$(sha256sum "$REQ_FILE" | awk '{print $1}')"
WHEELHOUSE="$CACHE_DIR/wheelhouse_$REQ_HASH"
REQ_STAMP="/content/miniconda3/envs/FMPCC/.fmpcc_requirements_hash"
FORCE_REINSTALL_REQS="${FORCE_REINSTALL_REQS:-0}"

mkdir -p "$PIP_CACHE" "$CACHE_DIR"
cd "$REPO"

if [ "$FORCE_REINSTALL_REQS" = "1" ]; then
  echo "FORCE_REINSTALL_REQS=1 -> forcing requirements reinstall"
  REQ_VALID=0
elif "$PY" - <<'PY'
import importlib
import sys

pkgs = [
    'torch', 'numpy', 'scipy', 'gym', 'gymnasium', 'gymnasium_robotics',
    'minari', 'wandb', 'mujoco', 'diffusers', 'transformers'
]

ok = True
for p in pkgs:
    try:
        importlib.import_module(p)
    except Exception as e:
        ok = False
        print(f'Missing/broken package: {p} ({type(e).__name__}: {e})')

import numpy
if int(numpy.__version__.split('.')[0]) >= 2:
    ok = False
    print(f'Invalid numpy version: {numpy.__version__}; expected 1.x for this workflow')

sys.exit(0 if ok else 2)
PY
then
  REQ_VALID=1
else
  REQ_VALID=0
fi

if [ "$REQ_VALID" = "1" ]; then
  if [ -f "$REQ_STAMP" ] && grep -Fxq "$REQ_HASH" "$REQ_STAMP"; then
    echo "Package validation passed and requirements hash unchanged; skipping reinstall"
  else
    echo "Package validation passed but requirements hash stamp missing/changed; updating stamp only"
    echo "$REQ_HASH" > "$REQ_STAMP"
  fi
else
  echo "Package validation failed; installing requirements"

  # Build or reuse a Drive-backed wheelhouse for faster future reinstalls.
  if [ ! -d "$WHEELHOUSE" ] || [ -z "$(ls -A "$WHEELHOUSE" 2>/dev/null || true)" ]; then
    echo "Building wheelhouse cache at: $WHEELHOUSE"
    mkdir -p "$WHEELHOUSE"
    PIP_CACHE_DIR="$PIP_CACHE" "$PIP" download -r requirements.txt -d "$WHEELHOUSE" || true
  else
    echo "Wheelhouse cache found: $WHEELHOUSE"
  fi

  WHEEL_COUNT="$(find "$WHEELHOUSE" -maxdepth 1 -type f \( -name '*.whl' -o -name '*.tar.gz' -o -name '*.zip' \) | wc -l)"
  if [ "$WHEEL_COUNT" -gt 0 ] && PIP_CACHE_DIR="$PIP_CACHE" "$PIP" install --no-index --find-links "$WHEELHOUSE" -r requirements.txt; then
    echo "Offline wheelhouse install succeeded"
  else
    echo "First run or incomplete wheelhouse -> running online install"
    PIP_CACHE_DIR="$PIP_CACHE" "$PIP" install -r requirements.txt

    # Backfill wheelhouse after a successful online install for faster future runs.
    PIP_CACHE_DIR="$PIP_CACHE" "$PIP" download -r requirements.txt -d "$WHEELHOUSE" || true
  fi

  # pip check may report known non-fatal issues on Colab (platform/extra metadata).
  PIP_CHECK_OUT="$("$PIP" check 2>&1 || true)"
  echo "$PIP_CHECK_OUT"

  UNEXPECTED_PIP_CHECK="$(echo "$PIP_CHECK_OUT" | grep -Ev '(^$|gurobipy .* is not supported on this platform|WARNING: typer .* does not provide the extra .all.)' || true)"
  if [ -n "$UNEXPECTED_PIP_CHECK" ]; then
    echo "Unexpected pip check issues found:"
    echo "$UNEXPECTED_PIP_CHECK"
    exit 1
  else
    echo "Only known non-fatal pip check warnings detected; continuing"
  fi

  echo "$REQ_HASH" > "$REQ_STAMP"
fi

# Quick sanity check
"$PY" - <<'PY'
import numpy, torch
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("python:", __import__("sys").executable)
PY



Missing/broken package: torch (ModuleNotFoundError: No module named 'torch')
Missing/broken package: scipy (ModuleNotFoundError: No module named 'scipy')
Missing/broken package: gymnasium (ModuleNotFoundError: No module named 'gymnasium')
Missing/broken package: gymnasium_robotics (ModuleNotFoundError: No module named 'gymnasium_robotics')
Missing/broken package: minari (ModuleNotFoundError: No module named 'minari')
Missing/broken package: mujoco (ModuleNotFoundError: No module named 'mujoco')
Missing/broken package: diffusers (ModuleNotFoundError: No module named 'diffusers')
Missing/broken package: transformers (ModuleNotFoundError: No module named 'transformers')
Invalid numpy version: 2.2.6; expected 1.x for this workflow
Package validation failed; installing requirements
Wheelhouse cache found: /content/drive/MyDrive/FMPCC/cache/wheelhouse_7f51b2eecd675d29f593a4ec70084ed4251266c758f7a4374628942b33efaa39
Looking in links: /content/drive/MyDrive/FMPCC/cache/wheelhouse_7f51b2eecd675

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## 7) Runtime Environment Variables

Includes W&B malformed service cleanup and Colab rendering settings.



In [7]:
import os

FMPCC = '/content/drive/MyDrive/FMPCC/FM-PCC'
D3IL_ROOT = f'{FMPCC}/d3il'
GYM_AV = f'{D3IL_ROOT}/environments/d3il/envs/gym_avoiding_env'

existing_pp = os.environ.get('PYTHONPATH', '')
parts = [FMPCC, D3IL_ROOT, GYM_AV]
if existing_pp:
    parts.append(existing_pp)

os.environ['FMPCC'] = FMPCC
os.environ['PYTHONPATH'] = ':'.join(parts)
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MPLBACKEND'] = 'agg'

for key in ('WANDB_SERVICE', 'WANDB__SERVICE'):
    os.environ.pop(key, None)

os.chdir(FMPCC)
print('cwd:', os.getcwd())
print('PYTHONPATH:', os.environ['PYTHONPATH'])



cwd: /content/drive/MyDrive/FMPCC/FM-PCC
PYTHONPATH: /content/drive/MyDrive/FMPCC/FM-PCC:/content/drive/MyDrive/FMPCC/FM-PCC/d3il:/content/drive/MyDrive/FMPCC/FM-PCC/d3il/environments/d3il/envs/gym_avoiding_env:/env/python


## 8) Optional W&B Login



In [8]:
import os
from pathlib import Path
import wandb

KEY_FILE = Path('/content/drive/MyDrive/FMPCC/.wandb_api_key')

if not KEY_FILE.exists():
    raise FileNotFoundError(
        f'Missing W&B key file: {KEY_FILE}. Create it with your API key on one line.'
    )

api_key = KEY_FILE.read_text(encoding='utf-8').strip()
if not api_key:
    raise ValueError(f'W&B key file is empty: {KEY_FILE}')

for k in ('WANDB_SERVICE', 'WANDB__SERVICE'):
    os.environ.pop(k, None)

wandb.finish()
os.environ['WANDB_MODE'] = 'online'
os.environ['WANDB_API_KEY'] = api_key

wandb.login(key=api_key, relogin=True)

print('W&B mode:', os.environ.get('WANDB_MODE'))
print('W&B key file:', KEY_FILE)
print('W&B key loaded:', f'***{api_key[-4:]}' if len(api_key) >= 4 else '***')



/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: llmmail2021 (llmmail2021-technical-university-of-munich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B mode: online
W&B key file: /content/drive/MyDrive/FMPCC/.wandb_api_key
W&B key loaded: ***KogM


## 9) Full Verification

Validates import chain with the exact env interpreter used for training.



In [9]:
%%bash
set -e

/content/miniconda3/envs/FMPCC/bin/python - <<'PY'
import importlib
import sys

pkgs = [
    'torch', 'numpy', 'scipy', 'gym', 'gymnasium', 'gymnasium_robotics',
    'minari', 'wandb', 'mujoco', 'diffusers', 'transformers'
]

ok = True
for p in pkgs:
    try:
        m = importlib.import_module(p)
        v = getattr(m, '__version__', 'unknown')
        print(f'{p:20s} {v}')
    except Exception as e:
        ok = False
        print(f'{p:20s} NOT IMPORTABLE ({type(e).__name__}: {e})')

import numpy, torch
print('numpy pinned:', numpy.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

major = int(numpy.__version__.split('.')[0])
if major >= 2:
    ok = False
    print('ERROR: numpy 2.x detected, expected 1.26.4 for this workflow')

if not ok:
    sys.exit(2)
PY



torch                2.2.2+cu121
numpy                1.26.4
scipy                1.13.1
gym                  0.26.2
gymnasium            0.29.1
gymnasium_robotics   1.2.4
minari               0.4.3
wandb                0.17.5
mujoco               2.3.7
diffusers            0.31.0
transformers         4.41.2
numpy pinned: 1.26.4
cuda available: True
device: Tesla T4


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


## 10) Dataset Preparation (Avoiding)

### Option A: Use existing zip from old DPCC path

Warning! : It searches the ~15Gb Zip in the DPCC Path, not this FMPCC Path!


This exits quickly if avoiding data already exists.



In [ ]:
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
AVOIDING_DATA="$REPO/d3il/environments/dataset/data/avoiding/data"
DATA_ZIP="/content/drive/MyDrive/DPCC/dpcc/d3il/environments/dataset/data/dataset.zip"

if [ -d "$AVOIDING_DATA" ] && [ "$(ls -A "$AVOIDING_DATA")" ]; then
  echo "avoiding data already present: $(ls "$AVOIDING_DATA" | wc -l) files"
  exit 0
fi

if [ ! -f "$DATA_ZIP" ]; then
  echo "dataset zip not found: $DATA_ZIP"
  echo "Skip this cell and use Option B below if needed."
  exit 1
fi

TMP="/content/avoiding_tmp"
rm -rf "$TMP"
mkdir -p "$TMP"
unzip -q "$DATA_ZIP" "avoiding/*" -d "$TMP"
mkdir -p "$REPO/d3il/environments/dataset/data/avoiding"
cp -r "$TMP/avoiding/." "$REPO/d3il/environments/dataset/data/avoiding/"
rm -rf "$TMP"

echo "avoiding dataset ready: $(ls "$AVOIDING_DATA" | wc -l) files"



### Option B: Download full D3IL dataset zip with gdown (only if Option A unavailable)



```
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
DATA_DIR="$REPO/d3il/environments/dataset/data"
ZIP_FILE="$DATA_DIR/dataset.zip"

if [ -f "$ZIP_FILE" ]; then
  echo "zip already exists: $ZIP_FILE"
  exit 0
fi

/content/miniconda3/envs/FMPCC/bin/pip install gdown -q
/content/miniconda3/envs/FMPCC/bin/python -m gdown \
  "https://drive.google.com/uc?id=1SQhbhzV85zf_ltnQ8Cbge2lsSWInxVa8" \
  -O "$ZIP_FILE"

echo "downloaded zip: $ZIP_FILE"



## 11) Smoke Test Train

Short check before full run.



# IN THIS TIME !!!


seed 6 for projection_eval.yaml

## ODE Solver Test

v4 code

time

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/v4/benchmark_ode_solvers_v4.py \
  --mode math \
  --vf-mode flow_matcher \
  --loadbase logs \
  --dataset avoiding-d3il \
  --diffusion-loadpath flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion \
  --diffusion-seed 6 \
  --device cuda \
  --n-trials 20 \
  --batch-size 4 \
  --horizon 8 \
  --steps 10 \
  --solver-spec legacy_euler,torchdiffeq:euler,legacy_midpoint,torchdiffeq:midpoint,legacy_rk4,torchdiffeq:rk4 \
  --plot \
  --output-dir FM_v3_ode_selectable_test/benchmark_outputs_v4/time/FMv3_VF_test_3_legacy_vs_3_torchdiffeq


[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 80000
ODE Benchmark V4 | mode=math | steps=10

[1/6] legacy:euler
  trial 000   190.306 ms
  trial 001   138.942 ms
  trial 002   153.790 ms
  trial 003   154.123 ms
  trial 004   638.167 ms
  trial 005   206.375 ms
  trial 006   122.497 ms
  trial 007   107.949 ms
  trial 008   117.953 ms
  trial 009   108.399 ms
  trial 010   107.973 ms
  trial 011   113.136 ms
  trial 012   123.722 ms
  trial 013   115.444 ms
  trial 014   107.558 ms
  trial 015   111.245 ms
  trial 016   109.047 ms
  trial 017   110.696 ms
  trial 018   108.215 ms
  trial 019   106.837 ms

[2/6] torchdiffeq:euler
  trial 

accuracy

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/v4/benchmark_ode_accuracy_v4.py \
  --mode math \
  --vf-mode flow_matcher \
  --loadbase logs \
  --dataset avoiding-d3il \
  --diffusion-loadpath flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion \
  --diffusion-seed 6 \
  --device cuda \
  --batch-size 4 \
  --horizon 8 \
  --steps 10 \
  --solver-spec legacy_euler,torchdiffeq:euler,legacy_midpoint,torchdiffeq:midpoint,legacy_rk4,torchdiffeq:rk4 \
  --plot \
  --output-dir FM_v3_ode_selectable_test/benchmark_outputs_v4/accuracy/FMv3_VF_test_3_legacy_vs_3_torchdiffeq


[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 80000
ODE Accuracy Audit V4 | Mode=math | Candidate Steps=10

[1] Generating ORACLE Ground Truth (Dopri5 @ 1e-10)...

    -> Simulating legacy:euler
      ✅ L2 Drift: 0.108495

    -> Simulating torchdiffeq:euler
      ✅ L2 Drift: 0.108495

    -> Simulating legacy:midpoint
      ✅ L2 Drift: 0.057627

    -> Simulating torchdiffeq:midpoint
      ✅ L2 Drift: 0.057627

    -> Simulating legacy:rk4
      ✅ L2 Drift: 0.040882

    -> Simulating torchdiffeq:rk4
      ✅ L2 Drift: 0.021411

================ FINAL V4 ACCURACY SUMMARY ================
 1. torchdiffeq:rk4        | Steps: 10 | Drift L2:

In [ ]:
Grid Search

accuracy- Grid

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python  FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/v4/grid_search_accuracy_v4.py \
  --mode math \
  --grid-batch 4 \
  --grid-steps 1,2,5,10,15,20 \
  --grid-horizon 8 \
  --solver-spec legacy_euler,torchdiffeq:euler,legacy_midpoint,torchdiffeq:midpoint,legacy_rk4,torchdiffeq:rk4 \
  --device cuda \
  --base-out FM_v3_ode_selectable_test/benchmark_outputs_v4/accuracy/ODE_Steps_1-20

=== Starting ODE Accuracy Grid Audit V4 [Mode: math] ===

[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 80000
ODE Accuracy Audit V4 | Mode=math | Candidate Steps=1

[1] Generating ORACLE Ground Truth (Dopri5 @ 1e-10)...

    -> Simulating legacy:euler
      ✅ L2 Drift: 0.748886

    -> Simulating torchdiffeq:euler
      ✅ L2 Drift: 0.748886

    -> Simulating legacy:midpoint
      ✅ L2 Drift: 0.441927

    -> Simulating torchdiffeq:midpoint
      ✅ L2 Drift: 0.441927

    -> Simulating legacy:rk4
      ✅ L2 Drift: 2.002118

    -> Simulating torchdiffeq:rk4
      ✅ L2 Drift: 1.642682

================ FINAL V4 ACCURACY SUMMARY ===========

time - Grid

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python  FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/v4/grid_search_benchmark_for_v4.py \
  --mode math \
  --grid-batch 4 \
  --grid-steps 1,2,5,10,15,20 \
  --grid-horizon 8 \
  --solver-spec legacy_euler,torchdiffeq:euler,legacy_midpoint,torchdiffeq:midpoint,legacy_rk4,torchdiffeq:rk4 \
  --device cuda \
  --base-out FM_v3_ode_selectable_test/benchmark_outputs_v4/time/ODE_Steps_1-20

=== Starting ODE Benchmark Grid Search V4 [Mode: math] ===

[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 80000
ODE Benchmark V4 | mode=math | steps=1

[1/6] legacy:euler
  trial 000    13.710 ms
  trial 001    13.976 ms
  trial 002    13.682 ms
  trial 003    13.411 ms
  trial 004    13.261 ms
  trial 005    13.112 ms
  trial 006    13.044 ms
  trial 007    13.659 ms
  trial 008    13.026 ms
  trial 009    13.022 ms
  trial 010    12.714 ms
  trial 011    12.985 ms
  trial 012    12.858 ms
  trial 013    12.734 ms
  trial 014    12.608 ms
  trial 015    13.329 ms
  trial 016    12.952 ms
  trial 017    12.537 ms
  trial 018    13.316 ms


old V3 codes

accraucy test

no need to run n-trails, same results, determined

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/v3/benchmark_ode_accuracy_v3.py \
  --mode math \
  --vf-mode flow_matcher \
  --loadbase logs \
  --dataset avoiding-d3il \
  --diffusion-loadpath flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion \
  --diffusion-seed 6 \
  --device cuda \
  --batch-size 4 \
  --horizon 8 \
  --steps 10 \
  --solver-spec legacy_euler,torchdiffeq:euler,legacy_midpoint,torchdiffeq:midpoint,legacy_rk4,torchdiffeq:rk4 \
  --plot \
  --output-dir FM_v3_ode_selectable_test/benchmark_outputs_v3/accuracy/FMv3_VF_test_3_legacy_vs_3_torchdiffeq


[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 80000
ODE Accuracy Audit V3 | Mode=math | Device=cuda | Candidate Steps=10 | Trials=1

[1] Generating ORACLE Ground Truth (ONCE)...
    ✅ Oracle Generated. (Terminal Norm: 8.1552)

    -> Simulating legacy:euler
      ✅ L2 Drift: 0.108495 | p50_ms: 136.87

    -> Simulating torchdiffeq:euler
      ✅ L2 Drift: 0.108495 | p50_ms: 140.23

    -> Simulating legacy:midpoint
      ✅ L2 Drift: 0.057627 | p50_ms: 250.38

    -> Simulating torchdiffeq:midpoint
      ✅ L2 Drift: 0.057627 | p50_ms: 253.23

    -> Simulating legacy:rk4
      ✅ L2 Drift: 0.040882 | p50_ms: 542.77

    -> Simulating torchd

grid search the ODE influences

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python  FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/v3/grid_search_accuracy_v3.py \
  --mode math \
  --grid-batch 4 \
  --grid-steps 1,2,5,10,15,20 \
  --grid-horizon 8 \
  --solver-spec legacy_euler,torchdiffeq:euler,legacy_midpoint,torchdiffeq:midpoint,legacy_rk4,torchdiffeq:rk4 \
  --device cuda \
  --base-out FM_v3_ode_selectable_test/benchmark_outputs_v3/accuracy/ODE_Steps_1-20

=== Starting ODE Accuracy Grid Search V3 [Mode: math] ===

[1/6] Running accuracy pass: acc_mode_math_h8_b4_s1

[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 80000
ODE Accuracy Audit V3 | Mode=math | Device=cuda | Candidate Steps=1 | Trials=1

[1] Generating ORACLE Ground Truth (ONCE)...
    ✅ Oracle Generated. (Terminal Norm: 8.1552)

    -> Simulating legacy:euler
      ✅ L2 Drift: 0.748886 | p50_ms: 16.19

    -> Simulating torchdiffeq:euler
      ✅ L2 Drift: 0.748886 | p50_ms: 17.40

    -> Simulating legacy:midpoint
      ✅ L2 Drift: 0.441927 | p50_ms: 34.12

    -> Simulating torchdiffeq:midpoint
      ✅ L2 Drift: 0.441927 | p50_ms:

### Old v1,v2,,v3 speed tests below

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python -m pip install torchdiffeq

only math competition

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/benchmark_ode_solvers_v3.py \
  --mode math \
  --vf-mode flow_matcher \
  --loadbase logs \
  --dataset avoiding-d3il \
  --diffusion-loadpath flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion \
  --diffusion-seed 6 \
  --device cuda \
  --n-trials 20 \
  --batch-size 4 \
  --horizon 8 \
  --steps 10 \
  --solver-spec legacy_euler,torchdiffeq:euler,legacy_midpoint,torchdiffeq:midpoint,legacy_rk4,torchdiffeq:rk4 \
  --plot \
  --output-dir FM_v3_ode_selectable_test/benchmark_outputs_v3/FMv3_VF_test_3_legacy_vs_3_torchdiffeq


[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 80000
ODE Benchmark V3 | mode=math | device=cuda | vf=flow_matcher | steps=10

[1/6] legacy:euler
  trial 000   114.178 ms
  trial 001   115.300 ms
  trial 002   108.984 ms
  trial 003   113.171 ms
  trial 004   125.561 ms
  trial 005   112.130 ms
  trial 006   115.854 ms
  trial 007   106.538 ms
  trial 008   109.189 ms
  trial 009   108.098 ms
  trial 010   108.415 ms
  trial 011   110.212 ms
  trial 012   110.656 ms
  trial 013   131.636 ms
  trial 014   109.375 ms
  trial 015   108.050 ms
  trial 016   109.493 ms
  trial 017   116.043 ms
  trial 018   110.110 ms
  trial 019   113.908 ms
 

test the touchdiffeq Parallel Capability

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/benchmark_ode_solvers_v3.py \
  --mode math \
  --vf-mode flow_matcher \
  --loadbase logs \
  --dataset avoiding-d3il \
  --diffusion-loadpath flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion \
  --diffusion-seed 6 \
  --device cuda \
  --n-trials 20 \
  --batch-size 1024 \
  --horizon 8 \
  --steps 10 \
  --solver-spec legacy_euler,torchdiffeq:euler,legacy_midpoint,torchdiffeq:midpoint,legacy_rk4,torchdiffeq:rk4 \
  --plot \
  --output-dir FM_v3_ode_selectable_test/benchmark_outputs_v3/FMv3_VF_test_3_legacy_vs_3_torchdiffeq_Batch_1024


[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 80000
ODE Benchmark V3 | mode=math | device=cuda | vf=flow_matcher | steps=10

[1/6] legacy:euler
  trial 000   343.049 ms
  trial 001   330.852 ms
  trial 002   508.250 ms
  trial 003   339.516 ms
  trial 004   427.930 ms
  trial 005   667.910 ms
  trial 006   338.431 ms
  trial 007   365.995 ms
  trial 008   340.559 ms
  trial 009   320.317 ms
  trial 010   326.439 ms
  trial 011   322.708 ms
  trial 012   322.222 ms
  trial 013   325.065 ms
  trial 014   322.911 ms
  trial 015   324.019 ms
  trial 016   324.425 ms
  trial 017   330.638 ms
  trial 018   329.516 ms
  trial 019   337.771 ms
 

what if CPU?

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/benchmark_ode_solvers_v3.py \
  --mode math \
  --vf-mode flow_matcher \
  --loadbase logs \
  --dataset avoiding-d3il \
  --diffusion-loadpath flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion \
  --diffusion-seed 6 \
  --device cpu \
  --n-trials 20 \
  --batch-size 4 \
  --horizon 8 \
  --steps 10 \
  --solver-spec legacy_euler,torchdiffeq:euler,legacy_midpoint,torchdiffeq:midpoint,legacy_rk4,torchdiffeq:rk4 \
  --plot \
  --output-dir FM_v3_ode_selectable_test/benchmark_outputs_v3/FMv3_VF_test_3_legacy_vs_3_torchdiffeq_Batch_4default_CPU


[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 80000
ODE Benchmark V3 | mode=math | device=cpu | vf=flow_matcher | steps=10

[1/6] legacy:euler
  trial 000   248.482 ms
  trial 001   246.213 ms
  trial 002   259.731 ms
  trial 003   256.073 ms
  trial 004   242.212 ms
  trial 005   242.938 ms
  trial 006   252.832 ms
  trial 007   288.305 ms
  trial 008   262.430 ms
  trial 009   273.445 ms
  trial 010   263.459 ms
  trial 011   229.001 ms
  trial 012   171.646 ms
  trial 013   166.344 ms
  trial 014   174.616 ms
  trial 015   168.034 ms
  trial 016   186.022 ms
  trial 017   189.970 ms
  trial 018   167.469 ms
  trial 019   164.832 ms
  

wire into production loop

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/benchmark_ode_solvers_v3.py \
  --mode production \
  --vf-mode flow_matcher \
  --loadbase logs \
  --dataset avoiding-d3il \
  --diffusion-loadpath flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion \
  --diffusion-seed 6 \
  --device cuda \
  --n-trials 20 \
  --batch-size 4 \
  --horizon 8 \
  --steps 10 \
  --solver-spec legacy_euler,torchdiffeq:euler,legacy_midpoint,torchdiffeq:midpoint,legacy_rk4,torchdiffeq:rk4 \
  --plot \
  --output-dir FM_v3_ode_selectable_test/benchmark_outputs_v3/Production_FMv3_VF_test_3_legacy_vs_3_torchdiffeq


[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 80000
ODE Benchmark V3 | mode=production | device=cuda | vf=flow_matcher | steps=10

[1/6] legacy:euler
  trial 000   149.991 ms
  trial 001   128.182 ms
  trial 002   116.269 ms
  trial 003   116.258 ms
  trial 004   115.141 ms
  trial 005   114.344 ms
  trial 006   113.191 ms
  trial 007   121.035 ms
  trial 008   114.339 ms
  trial 009   109.917 ms
  trial 010   130.137 ms
  trial 011   112.604 ms
  trial 012   115.806 ms
  trial 013   115.271 ms
  trial 014   110.715 ms
  trial 015   111.626 ms
  trial 016   114.761 ms
  trial 017   112.887 ms
  trial 018   110.640 ms
  trial 019   130.68

Grid Search the ODE step.

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python  FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/grid_search_benchmark_for_v3.py \
  --mode math \
  --grid-batch 4 \
  --grid-steps 1,2,5,10,15,20 \
  --grid-horizon 8 \
  --solver-spec legacy_euler,torchdiffeq:euler,legacy_midpoint,torchdiffeq:midpoint,legacy_rk4,torchdiffeq:rk4 \
  --device cuda \
  --base-out FM_v3_ode_selectable_test/benchmark_outputs_v3/ODE_Steps_1-20

=== Starting ODE Benchmark Grid Search V3 [Mode: math] ===

[1/6] Running combination: mode_math_h8_b4_s1_pure_gpu

[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 80000
ODE Benchmark V3 | mode=math | device=cuda | vf=flow_matcher | steps=1

[1/6] legacy:euler
  trial 000    14.581 ms
  trial 001    15.998 ms
  trial 002    14.681 ms
  trial 003    14.760 ms
  trial 004    15.656 ms
  trial 005    17.843 ms
  trial 006    17.131 ms
  trial 007    16.797 ms
  trial 008    18.883 ms
  trial 009    17.109 ms
  trial 010    15.273 ms
  trial 011    14.908 ms
  trial 012    14.328 ms
  trial 013    14.243 ms
  trial 014    14.250 ms
  trial 015 

  --grid-batch 1,2,4,8,16,32 \


In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python  FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/grid_search_benchmark_for_v3.py \
  --mode math \
  --grid-batch 1,2,4,8,16,32 \
  --grid-steps 10 \
  --grid-horizon 8 \
  --solver-spec legacy_euler,torchdiffeq:euler,legacy_midpoint,torchdiffeq:midpoint,legacy_rk4,torchdiffeq:rk4 \
  --device cuda \
  --base-out FM_v3_ode_selectable_test/benchmark_outputs_v3/batch1_32

=== Starting ODE Benchmark Grid Search V3 [Mode: math] ===

[1/6] Running combination: mode_math_h8_b1_s10_pure_gpu

[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 80000
ODE Benchmark V3 | mode=math | device=cuda | vf=flow_matcher | steps=10

[1/6] legacy:euler
  trial 000   169.617 ms
  trial 001   151.824 ms
  trial 002   146.884 ms
  trial 003   144.408 ms
  trial 004   153.980 ms
  trial 005   155.392 ms
  trial 006   163.226 ms
  trial 007   197.437 ms
  trial 008   164.412 ms
  trial 009   185.276 ms
  trial 010   194.633 ms
  trial 011   209.112 ms
  trial 012   136.947 ms
  trial 013   139.590 ms
  trial 014   120.108 ms
  trial 01

on real VF

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/benchmark_ode_solvers_v2.py \
  --vf-mode flow_matcher \
  --loadbase logs \
  --dataset avoiding-d3il \
  --diffusion-loadpath flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion \
  --diffusion-seed 6 \
  --n-trials 20 \
  --batch-size 4 \
  --horizon 8 \
  --steps 10 \
  --device cuda \
  --solver-spec legacy_euler,torchdiffeq:euler,torchdiffeq:midpoint,torchdiffeq:rk4 \
  --plot \
  --output-dir FM_v3_ode_selectable_test/benchmark_outputs_v2/FMv3_VF_test_euler_vs_tourcheuler_midpoint_rk4

compare the Native vs Touchdiffeq

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/benchmark_ode_solvers_v2.py \
--vf-mode flow_matcher \
--loadbase logs \
--dataset avoiding-d3il \
--diffusion-loadpath flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion \
--diffusion-seed 6 \
--n-trials 20 \
--batch-size 4 \
--horizon 8 \
--steps 10 \
--device cuda \
--solver-spec legacy_euler,torchdiffeq:euler,legacy_midpoint,torchdiffeq:midpoint,legacy_rk4,torchdiffeq:rk4,legacy_dopri5,torchdiffeq:dopri5 \
--plot \
--output-dir FM_v3_ode_selectable_test/benchmark_outputs_v2/FMv3_VF_test_4_legacy_vs_4_torchdiffeq


### Grid Search

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/grid_search_benchmark_for_v2.py \
  --vf-mode flow_matcher \
  --loadbase logs \
  --dataset avoiding-d3il \
  --diffusion-loadpath flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion \
  --diffusion-seed 6 \
  --n-trials 20 \
  --device cuda \
  --solver-spec legacy_euler,torchdiffeq:euler,torchdiffeq:midpoint,torchdiffeq:rk4,torchdiffeq:dopri5 \
  --grid-horizon 8 \
  --grid-batch 4 \
  --grid-steps 1,3,5,7,9,11,13,15,17,19 \
  --base-out FM_v3_ode_selectable_test/benchmark_outputs_v2/GridSearch_MetaAnalysis

### Search the Bathsize influence

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/Benchmark_ode_solver_Tests/grid_search_benchmark_for_v2.py \
  --vf-mode flow_matcher \
  --loadbase logs \
  --dataset avoiding-d3il \
  --diffusion-loadpath flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion \
  --solver-spec legacy_euler,torchdiffeq:euler,torchdiffeq:midpoint,torchdiffeq:rk4,torchdiffeq:dopri5 \
  --grid-horizon 8 \
  --grid-steps 10 \
  --grid-batch 4,16,32,64 \
  --base-out FM_v3_ode_selectable_test/benchmark_outputs_v2/Batch_Capacity_h8_b4-256_s10

simple fast test (simple_task_v1_cold_start_touchdiffeq_overhead_test & simple_task_v2_warm_up_run_overhead_test)

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/benchmark_ode_solvers.py \
  --n-trials 20 \
  --batch-size 1024 \
  --state-dim 64 \
  --steps 50 \
  --solver-spec legacy_euler,torchdiffeq:euler,torchdiffeq:midpoint,torchdiffeq:rk4 \
  --plot \
  --output-dir FM_v3_ode_selectable_test/benchmark_outputs/overhead_test

Hard_NN_VF_cuda (Batch = 1)
&
Hard_NN_VF_cuda_Batch1024

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/benchmark_ode_solvers.py \
  --vf-mode neural --batch-size 1024 --state-dim 8 --steps 20 \
  --solver-spec legacy_euler,torchdiffeq:euler,torchdiffeq:midpoint,torchdiffeq:rk4 \
  --plot \
  --device cuda \
  --output-dir FM_v3_ode_selectable_test/benchmark_outputs/Hard_NN_VF_cuda_Batch1024

Hard_NN_VF_cpu_Batch100

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/benchmark_ode_solvers.py \
  --vf-mode neural --batch-size 100 --state-dim 8 --steps 20 \
  --solver-spec legacy_euler,torchdiffeq:euler,torchdiffeq:midpoint,torchdiffeq:rk4 \
  --plot \
  --device cpu \
  --output-dir FM_v3_ode_selectable_test/benchmark_outputs/Hard_NN_VF_cpu_Batch100

## 12) Full Train

Real-time streaming via `!python`.



In [ ]:
#!/content/miniconda3/envs/FMPCC/bin/python scripts/train.py --seeds 6 --num-seeds 1 --use-wandb --wandb-project FMPCC

#!/content/miniconda3/envs/FMPCC/bin/python FM_test/train_FM.py --seeds 7 --num-seeds 1 --use-wandb --wandb-project FMPCC
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/train_flow_matching_v3_ode_selectable.py --seeds 6 --num-seeds 1 --use-wandb --wandb-project FMPCC

[ train ] Seed source: cli --seeds
[ train ] Training seeds: [6]
[ utils/setup ] Made savepath: logs/avoiding-d3il/flow_matching_v3_ode_selectable_aw10_ODE20/H8_K20_Dmodels.diffusion.GaussianDiffusion/6
[ train ] Saved seed manifest to: logs/avoiding-d3il/flow_matching_v3_ode_selectable_aw10_ODE20/H8_K20_Dmodels.diffusion.GaussianDiffusion/seeds_config.json
wandb: Currently logged in as: llmmail2021 (llmmail2021-technical-university-of-munich). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.26.0 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in /content/drive/MyDrive/FMPCC/FM-PCC/wandb/run-20260422_202139-3dgsniur
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run avoiding-d3il-seed-6
wandb: ⭐️ View project at https://wandb.ai/llmmail2021-technical-university-of-munich/FMPCC
wandb: 🚀 View run at https://wandb.ai/llmmail2021-technical-universi

## 13) Resume Training (Optional)



In [ ]:
#!/content/miniconda3/envs/FMPCC/bin/python scripts/train.py --seeds 5 --auto-resume --use-wandb --wandb-project FMPCC
!/content/miniconda3/envs/FMPCC/bin/python FM_test/train_FM.py --seeds 7 --auto-resume --use-wandb --wandb-project FMPCC

In [10]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/train_flow_matching_v3_ode_selectable.py --auto-resume --seeds 6 --num-seeds 1 --use-wandb --wandb-project FMPCC

[ train ] Seed source: cli --seeds
[ train ] Training seeds: [6]
[ train ] Saved seed manifest to: logs/avoiding-d3il/flow_matching_v3_ode_selectable_aw10_ODE20/H8_K20_Dmodels.diffusion.GaussianDiffusion/seeds_config.json
wandb: Currently logged in as: llmmail2021 (llmmail2021-technical-university-of-munich). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.26.1 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in /content/drive/MyDrive/FMPCC/FM-PCC/wandb/run-20260423_215929-waewbm9o
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run avoiding-d3il-seed-6
wandb: ⭐️ View project at https://wandb.ai/llmmail2021-technical-university-of-munich/FMPCC
wandb: 🚀 View run at https://wandb.ai/llmmail2021-technical-university-of-munich/FMPCC/runs/waewbm9o

[utils/config ] Config: <class 'flow_matcher_v3_ode_selectable.datasets.sequence.SequenceDataset'>
    d

## 14) Evaluation and Results



        'diffusion_loadpath': 'f:flow_matching_v3_ode_selectable_diffusion_timestep_threshold_1/H{horizon}_K{n_diffusion_steps}_D{diffusion}',


Remember to edit the yaml in /config to choose seeds
and
must write_to_file: True

In [ ]:
#!/content/miniconda3/envs/FMPCC/bin/python scripts/eval.py
!/content/miniconda3/envs/FMPCC/bin/python FM_test/eval_FM.py


In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/eval_flow_matching_v3_ode_selectable.py

pybullet build time: Nov 28 2023 23:45:17
[ utils/setup ] Made savepath: logs/avoiding-d3il/plans/flow_matching_v3_ode_selectable_diffusion_aw10_ODE20/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_v3_ode_selectable_aw10_ODE20/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 80000

Final IK error (78 iterations):  7.2858622740627195e-06
Final IK error (0 iterations):  7.2858622740627195e-06
------------------------Running avoiding-d3il - top-right-hard - dpcc-r (6)----------------------------
Success rate: 0.0
Constraints satisfied: 0.0
Success rate (goal and constraints): 0.0
Avg number of steps: 0.00 +- 0.00
Avg number of constraint violations

### Load Results

If the process crashed change the yaml to resume at crash point.

Example Save Path : `logs/avoiding-d3il/plans/H8_K20_Dmodels.GaussianDiffusion/0/results/`

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_test/load_results_FM.py


In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_ode_selectable_test/load_results_flow_matching_v3_ode_selectable.py

## 15) Visualization



In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python scripts/visualize_data_constraints.py



## 16) Archive Logs to Drive



In [ ]:
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
STAMP="$(date +%Y%m%d_%H%M%S)"
OUT="/content/drive/MyDrive/FMPCC/runs_snapshot/$STAMP"

mkdir -p "$OUT"
if [ -d "$REPO/logs" ]; then
  cp -r "$REPO/logs" "$OUT/"
fi
if [ -d "$REPO/wandb" ]; then
  cp -r "$REPO/wandb" "$OUT/"
fi

echo "snapshot saved: $OUT"


# misc

## Load the Training Results (NOT USING CONDA)

```
# ============================================================
# WHAT WE ARE PLOTTING (4 curves across 2 subplots):
#
#   Left subplot — "Total Loss":
#     ✅ training_losses  (pkl col "training_losses"  + wandb "loss")
#     ✅ test_losses       (pkl col "test_losses"      + wandb "loss_test")
#
#   Right subplot — "a0 Loss (Action Prediction)":
#     ✅ training_a0_losses (pkl col "training_a0_losses" + wandb "a0_loss")
#     ✅ test_a0_losses     (pkl col "test_a0_losses"     + wandb "a0_loss_test")
#
# WHAT WE ARE NOT PLOTTING:
#     ❌ diffusion_loss       (available in wandb log, not plotted)
#     ❌ diffusion_loss_test   (available in wandb log, not plotted)
#     ❌ lr (learning rate)    (available in wandb log, not plotted)
#
# DATA SOURCES:
#     - losses.pkl  → late training data (~step 40000+)
#     - WANDB_LOG_TEXT (hardcoded console paste) → epochs 0-72, steps 999-72193
#     - combine() only adds wandb steps NOT already in pkl
# ============================================================

FM


In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

# ── Fallback Unpickler ──
class FallbackUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if 'flow_matcher' in module:
            return type(name, (), {})
        return super().find_class(module, name)

# ── Target directory ──
target_dir = '/content/drive/MyDrive/FMPCC/FM-PCC/logs/avoiding-d3il/flow_matching/H8_K20_Dmodels.diffusion.GaussianDiffusion/6'

path_parts = target_dir.rstrip('/').split('/')
identifier = '__'.join(path_parts[3:]).replace('Dmodels.diffusion.', '').replace('.', '_')
print(f"Run identifier: {identifier}\n")

all_files = [
    f for f in os.listdir(target_dir)
    if os.path.isfile(os.path.join(target_dir, f)) and ':' not in f
]

print(f"Found {len(all_files)} files in: {target_dir}\n")

for file_name in all_files:
    path = os.path.join(target_dir, file_name)
    print(f"{'='*30} FILE: {file_name} {'='*30}")

    try:
        # Handle NumPy Archive (.npz)
        if file_name.endswith('.npz'):
            with np.load(path, allow_pickle=True) as data:
                for key in data.files:
                    arr = data[key]
                    print(f"Key: {key:<20} | Shape: {str(arr.shape):<12} | Dtype: {arr.dtype}")
                    if arr.size > 0:
                        print(arr if arr.size < 10 else f"First 5: {arr.flatten()[:5]}...")

        # Handle Pickle Files (.pkl) — always use FallbackUnpickler
        elif file_name.endswith('.pkl'):
            with open(path, 'rb') as f:
                content = FallbackUnpickler(f).load()

            if isinstance(content, dict):
                for k, v in content.items():
                    print(f"{k:<25}: {v}")
            elif hasattr(content, '__dict__'):
                print(f"  Type: {type(content).__name__}")
                for k, v in vars(content).items():
                    print(f"  {k}: {v}")
            elif isinstance(content, (list, np.ndarray)):
                arr_content = np.array(content)
                print(f"Data Type: {type(content)} | Length: {len(arr_content)}")
                print(f"Last Values: {arr_content[-5:]}")
            else:
                print(content)

            # ── Plot loss curves if this is losses.pkl ──
            if file_name == 'losses.pkl' and isinstance(content, dict):
                fig, axes = plt.subplots(1, 2, figsize=(14, 5))

                for key, label, color in [
                    ('training_losses', 'Training Loss', 'tab:blue'),
                    ('test_losses', 'Test Loss', 'tab:orange'),
                ]:
                    if key in content:
                        data = np.array(content[key])
                        axes[0].plot(data[:, 0], data[:, 1], label=label, color=color, alpha=0.8)
                axes[0].set_xlabel('Step')
                axes[0].set_ylabel('Loss')
                axes[0].set_title('Total Loss')
                axes[0].legend()
                axes[0].grid(True, alpha=0.3)

                for key, label, color in [
                    ('training_a0_losses', 'Training a0 Loss', 'tab:blue'),
                    ('test_a0_losses', 'Test a0 Loss', 'tab:orange'),
                ]:
                    if key in content:
                        data = np.array(content[key])
                        axes[1].plot(data[:, 0], data[:, 1], label=label, color=color, alpha=0.8)
                axes[1].set_xlabel('Step')
                axes[1].set_ylabel('Loss')
                axes[1].set_title('a0 Loss (Action Prediction)')
                axes[1].legend()
                axes[1].grid(True, alpha=0.3)

                plt.suptitle(f'Training — {identifier}', fontsize=13, fontweight='bold')
                plt.tight_layout()

                save_name = f'{identifier}__loss_curves.png'
                save_path = os.path.join(target_dir, save_name)
                plt.savefig(save_path, dpi=150)
                plt.show()
                print(f"\n✅ Loss curves saved to {save_path}")

        # Handle PyTorch Weights (.pt)
        elif file_name.endswith('.pt'):
            import torch
            checkpoint = torch.load(path, map_location='cpu')
            if isinstance(checkpoint, dict):
                print(f"Keys in Checkpoint: {list(checkpoint.keys())}")
            else:
                print("Raw Tensor/Object loaded.")

    except Exception as e:
        print(f"FAILED TO LOAD {file_name}: {e}")

    print("-" * 80)

DPCC

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

# ── Fallback Unpickler ──
class FallbackUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if 'flow_matcher' in module:
            return type(name, (), {})
        return super().find_class(module, name)

# ── Target directory ──
target_dir = '/content/drive/MyDrive/FMPCC/FM-PCC/logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/5'

path_parts = target_dir.rstrip('/').split('/')
identifier = '__'.join(path_parts[3:]).replace('Dmodels.diffusion.', '').replace('.', '_')
print(f"Run identifier: {identifier}\n")

all_files = [
    f for f in os.listdir(target_dir)
    if os.path.isfile(os.path.join(target_dir, f)) and ':' not in f
]

print(f"Found {len(all_files)} files in: {target_dir}\n")

for file_name in all_files:
    path = os.path.join(target_dir, file_name)
    print(f"{'='*30} FILE: {file_name} {'='*30}")

    try:
        # Handle NumPy Archive (.npz)
        if file_name.endswith('.npz'):
            with np.load(path, allow_pickle=True) as data:
                for key in data.files:
                    arr = data[key]
                    print(f"Key: {key:<20} | Shape: {str(arr.shape):<12} | Dtype: {arr.dtype}")
                    if arr.size > 0:
                        print(arr if arr.size < 10 else f"First 5: {arr.flatten()[:5]}...")

        # Handle Pickle Files (.pkl) — always use FallbackUnpickler
        elif file_name.endswith('.pkl'):
            with open(path, 'rb') as f:
                content = FallbackUnpickler(f).load()

            if isinstance(content, dict):
                for k, v in content.items():
                    print(f"{k:<25}: {v}")
            elif hasattr(content, '__dict__'):
                print(f"  Type: {type(content).__name__}")
                for k, v in vars(content).items():
                    print(f"  {k}: {v}")
            elif isinstance(content, (list, np.ndarray)):
                arr_content = np.array(content)
                print(f"Data Type: {type(content)} | Length: {len(arr_content)}")
                print(f"Last Values: {arr_content[-5:]}")
            else:
                print(content)

            # ── Plot loss curves if this is losses.pkl ──
            if file_name == 'losses.pkl' and isinstance(content, dict):
                fig, axes = plt.subplots(1, 2, figsize=(14, 5))

                for key, label, color in [
                    ('training_losses', 'Training Loss', 'tab:blue'),
                    ('test_losses', 'Test Loss', 'tab:orange'),
                ]:
                    if key in content:
                        data = np.array(content[key])
                        axes[0].plot(data[:, 0], data[:, 1], label=label, color=color, alpha=0.8)
                axes[0].set_xlabel('Step')
                axes[0].set_ylabel('Loss')
                axes[0].set_title('Total Loss')
                axes[0].legend()
                axes[0].grid(True, alpha=0.3)

                for key, label, color in [
                    ('training_a0_losses', 'Training a0 Loss', 'tab:blue'),
                    ('test_a0_losses', 'Test a0 Loss', 'tab:orange'),
                ]:
                    if key in content:
                        data = np.array(content[key])
                        axes[1].plot(data[:, 0], data[:, 1], label=label, color=color, alpha=0.8)
                axes[1].set_xlabel('Step')
                axes[1].set_ylabel('Loss')
                axes[1].set_title('a0 Loss (Action Prediction)')
                axes[1].legend()
                axes[1].grid(True, alpha=0.3)

                plt.suptitle(f'Training — {identifier}', fontsize=13, fontweight='bold')
                plt.tight_layout()

                save_name = f'{identifier}__loss_curves.png'
                save_path = os.path.join(target_dir, save_name)
                plt.savefig(save_path, dpi=150)
                plt.show()
                print(f"\n✅ Loss curves saved to {save_path}")

        # Handle PyTorch Weights (.pt)
        elif file_name.endswith('.pt'):
            import torch
            checkpoint = torch.load(path, map_location='cpu')
            if isinstance(checkpoint, dict):
                print(f"Keys in Checkpoint: {list(checkpoint.keys())}")
            else:
                print("Raw Tensor/Object loaded.")

    except Exception as e:
        print(f"FAILED TO LOAD {file_name}: {e}")

    print("-" * 80)

## Load with train data pkl, and print into output


In [ ]:
import pickle

pkl_path = '/content/drive/MyDrive/FMPCC/FM-PCC/logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6/losses.pkl'

with open(pkl_path, 'rb') as f:
    data = pickle.load(f)

data_fm_v3 = data
data_fm_v3['_label'] = 'FM_v3'
data_fm_v3

In [ ]:
import pickle

pkl_path = '/content/drive/MyDrive/FMPCC/FM-PCC/logs/avoiding-d3il/flow_matching_v2/H8_K20_Dmodels.diffusion.GaussianDiffusion/6/losses.pkl'

with open(pkl_path, 'rb') as f:
    data = pickle.load(f)

data_fm_v2 = data
data_fm_v2['_label'] = 'FM_v2'
data_fm_v2

In [ ]:
import pickle

pkl_path = '/content/drive/MyDrive/FMPCC/FM-PCC/logs/avoiding-d3il/flow_matching_hp_tune1/H8_K20_Dmodels.diffusion.GaussianDiffusion/6/losses.pkl'

with open(pkl_path, 'rb') as f:
    data = pickle.load(f)

data_fm_hp = data
data_fm_hp['_label'] = 'FM HP Tune1'
data_fm_hp

In [ ]:
import pickle

pkl_path = '/content/drive/MyDrive/FMPCC/FM-PCC/logs/avoiding-d3il/flow_matching/H8_K20_Dmodels.diffusion.GaussianDiffusion/6/losses.pkl'

with open(pkl_path, 'rb') as f:
    data = pickle.load(f)

data_fm = data
data_fm['_label'] = 'FM Baseline'
data_fm

Caution: for DPCC Seed 6 Training Logs, is patched from Wandb because the data lost in resume training and bugs in pkl logging(fixed now). So the under code is also different

In [ ]:
import os, re, pickle
import numpy as np

target_dir = '/content/drive/MyDrive/FMPCC/FM-PCC/logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/6'

class FallbackUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if "flow_matcher" in module:
            return type(name, (), {})
        return super().find_class(module, name)

def load_losses_pkl(path):
    with open(path, "rb") as f:
        d = FallbackUnpickler(f).load()
    out = {}
    for k, v in d.items():
        arr = np.array(v)
        if arr.ndim == 2 and arr.shape[1] == 2:
            out[k] = arr.astype(float)
    return out

pkl = load_losses_pkl(os.path.join(target_dir, "losses.pkl"))

# ── Paste W&B console output ──
WANDB_LOG_TEXT = r"""
Epoch 0: 100% 1000/1000 [01:21<00:00, 12.31it/s, a0_loss=0.0414, a0_loss_test=0.112, diffusion_loss=0.853, loss=0.426, loss_test=0.751, lr=0.0001, step=999]
2026-03-20 15:47:35Initial test loss:   0.7509, a0 loss:   0.1115
2026-03-20 15:50:14Epoch 1: 100% 1000/1000 [01:19<00:00, 12.57it/s, a0_loss=0.0371, a0_loss_test=0.0343, diffusion_loss=0.483, loss=0.241, loss_test=0.393, lr=0.0001, step=1999]
2026-03-20 15:51:34Epoch 2: 100% 1000/1000 [01:20<00:00, 12.44it/s, a0_loss=0.0183, a0_loss_test=0.0256, diffusion_loss=0.195, loss=0.0975, loss_test=0.196, lr=9.99e-5, step=2999]
2026-03-20 15:52:56Epoch 3: 100% 1000/1000 [01:21<00:00, 12.26it/s, a0_loss=0.0145, a0_loss_test=0.0219, diffusion_loss=0.157, loss=0.0783, loss_test=0.134, lr=9.98e-5, step=3999]
2026-03-20 15:54:20Epoch 4: 100% 1000/1000 [01:25<00:00, 11.65it/s, a0_loss=0.0109, a0_loss_test=0.0212, diffusion_loss=0.142, loss=0.0712, loss_test=0.105, lr=9.96e-5, step=4999]
2026-03-20 15:55:45Epoch 5: 100% 1000/1000 [01:23<00:00, 12.00it/s, a0_loss=0.0195, a0_loss_test=0.0219, diffusion_loss=0.199, loss=0.0993, loss_test=0.0995, lr=9.94e-5, step=5999]
2026-03-20 15:57:05Epoch 6: 100% 1000/1000 [01:21<00:00, 12.32it/s, a0_loss=0.00851, a0_loss_test=0.0197, diffusion_loss=0.0818, loss=0.0409, loss_test=0.0904, lr=9.91e-5, step=6999]
2026-03-20 15:58:29Epoch 7: 100% 1000/1000 [01:23<00:00, 12.05it/s, a0_loss=0.0373, a0_loss_test=0.0188, diffusion_loss=0.323, loss=0.161, loss_test=0.0845, lr=9.88e-5, step=7999]
2026-03-20 15:59:49Epoch 8: 100% 1000/1000 [01:21<00:00, 12.28it/s, a0_loss=0.0197, a0_loss_test=0.0189, diffusion_loss=0.133, loss=0.0664, loss_test=0.0811, lr=9.84e-5, step=8999]
2026-03-20 16:01:11Epoch 9: 100% 1000/1000 [01:20<00:00, 12.39it/s, a0_loss=0.0206, a0_loss_test=0.0155, diffusion_loss=0.228, loss=0.114, loss_test=0.0666, lr=9.8e-5, step=9999]
2026-03-20 16:02:32Epoch 10: 100% 1000/1000 [01:21<00:00, 12.32it/s, a0_loss=0.00351, a0_loss_test=0.014, diffusion_loss=0.0464, loss=0.0232, loss_test=0.0603, lr=9.75e-5, step=10999]
2026-03-20 16:03:50Epoch 11: 100% 1000/1000 [01:19<00:00, 12.65it/s, a0_loss=0.0101, a0_loss_test=0.0152, diffusion_loss=0.0833, loss=0.0416, loss_test=0.0617, lr=9.7e-5, step=11999]
2026-03-20 16:05:12Epoch 12: 100% 1000/1000 [01:21<00:00, 12.25it/s, a0_loss=0.048, a0_loss_test=0.0155, diffusion_loss=0.41, loss=0.205, loss_test=0.0623, lr=9.64e-5, step=12999]
2026-03-20 16:06:34Epoch 13: 100% 1000/1000 [01:21<00:00, 12.28it/s, a0_loss=0.0183, a0_loss_test=0.0147, diffusion_loss=0.123, loss=0.0613, loss_test=0.0576, lr=9.58e-5, step=13999]
2026-03-20 16:07:56Epoch 14: 100% 1000/1000 [01:21<00:00, 12.20it/s, a0_loss=0.0175, a0_loss_test=0.0134, diffusion_loss=0.0995, loss=0.0498, loss_test=0.0547, lr=9.51e-5, step=14999]
2026-03-20 16:09:17Epoch 15: 100% 1000/1000 [01:21<00:00, 12.34it/s, a0_loss=0.0108, a0_loss_test=0.015, diffusion_loss=0.152, loss=0.0758, loss_test=0.0599, lr=9.44e-5, step=15999]
2026-03-20 16:10:39Epoch 16: 100% 1000/1000 [01:21<00:00, 12.32it/s, a0_loss=0.012, a0_loss_test=0.0105, diffusion_loss=0.104, loss=0.052, loss_test=0.0456, lr=9.37e-5, step=16999]
2026-03-20 16:11:59Epoch 17: 100% 1000/1000 [01:19<00:00, 12.53it/s, a0_loss=0.0431, a0_loss_test=0.0134, diffusion_loss=0.222, loss=0.111, loss_test=0.0541, lr=9.29e-5, step=17999]
2026-03-20 16:13:19Epoch 18: 100% 1000/1000 [01:20<00:00, 12.40it/s, a0_loss=0.00677, a0_loss_test=0.0133, diffusion_loss=0.0682, loss=0.0341, loss_test=0.0499, lr=9.21e-5, step=18999]
2026-03-20 16:14:41Epoch 19: 100% 1000/1000 [01:23<00:00, 11.92it/s, a0_loss=0.00855, a0_loss_test=0.0115, diffusion_loss=0.0669, loss=0.0334, loss_test=0.0485, lr=9.12e-5, step=2e+4]
2026-03-20 16:16:06Epoch 20: 100% 1000/1000 [01:22<00:00, 12.09it/s, a0_loss=0.0513, a0_loss_test=0.012, diffusion_loss=0.297, loss=0.149, loss_test=0.0467, lr=9.03e-5, step=20999]
2026-03-20 16:17:30Epoch 21: 100% 1000/1000 [01:23<00:00, 11.93it/s, a0_loss=0.0219, a0_loss_test=0.0132, diffusion_loss=0.162, loss=0.0812, loss_test=0.051, lr=8.93e-5, step=21999]
2026-03-20 16:18:54Epoch 22: 100% 1000/1000 [01:24<00:00, 11.77it/s, a0_loss=0.00705, a0_loss_test=0.0137, diffusion_loss=0.0584, loss=0.0292, loss_test=0.0504, lr=8.83e-5, step=22999]
2026-03-20 16:20:16Epoch 23: 100% 1000/1000 [01:23<00:00, 12.05it/s, a0_loss=0.00206, a0_loss_test=0.0132, diffusion_loss=0.0291, loss=0.0146, loss_test=0.0471, lr=8.73e-5, step=23999]
2026-03-20 16:21:41Epoch 24: 100% 1000/1000 [01:24<00:00, 11.89it/s, a0_loss=0.00462, a0_loss_test=0.0135, diffusion_loss=0.044, loss=0.022, loss_test=0.0493, lr=8.62e-5, step=24999]
2026-03-20 16:23:07Epoch 25: 100% 1000/1000 [01:25<00:00, 11.64it/s, a0_loss=0.0458, a0_loss_test=0.0107, diffusion_loss=0.292, loss=0.146, loss_test=0.043, lr=8.51e-5, step=25999]
2026-03-20 16:24:31Epoch 26: 100% 1000/1000 [01:24<00:00, 11.79it/s, a0_loss=0.013, a0_loss_test=0.0117, diffusion_loss=0.162, loss=0.0809, loss_test=0.0452, lr=8.39e-5, step=26999]
2026-03-20 16:25:57Epoch 27: 100% 1000/1000 [01:25<00:00, 11.65it/s, a0_loss=0.0173, a0_loss_test=0.00904, diffusion_loss=0.101, loss=0.0505, loss_test=0.0384, lr=8.27e-5, step=27999]
2026-03-20 16:27:19Epoch 28: 100% 1000/1000 [01:21<00:00, 12.20it/s, a0_loss=0.00377, a0_loss_test=0.0109, diffusion_loss=0.0308, loss=0.0154, loss_test=0.0423, lr=8.15e-5, step=28999]
2026-03-20 16:28:42Epoch 29: 100% 1000/1000 [01:21<00:00, 12.28it/s, a0_loss=0.00501, a0_loss_test=0.0113, diffusion_loss=0.0453, loss=0.0226, loss_test=0.0448, lr=8.03e-5, step=3e+4]
2026-03-20 16:30:00Epoch 30: 100% 1000/1000 [01:19<00:00, 12.61it/s, a0_loss=0.00641, a0_loss_test=0.011, diffusion_loss=0.0452, loss=0.0226, loss_test=0.0423, lr=7.9e-5, step=30999]
2026-03-20 16:31:22Epoch 31: 100% 1000/1000 [01:21<00:00, 12.29it/s, a0_loss=0.0214, a0_loss_test=0.0114, diffusion_loss=0.112, loss=0.0559, loss_test=0.045, lr=7.77e-5, step=31999]
2026-03-20 16:32:46Epoch 32: 100% 1000/1000 [01:23<00:00, 11.98it/s, a0_loss=0.00629, a0_loss_test=0.0127, diffusion_loss=0.0639, loss=0.0319, loss_test=0.0469, lr=7.64e-5, step=32999]
2026-03-20 16:34:06Epoch 33: 100% 1000/1000 [01:20<00:00, 12.36it/s, a0_loss=0.00533, a0_loss_test=0.0103, diffusion_loss=0.0719, loss=0.0359, loss_test=0.0411, lr=7.5e-5, step=33999]
2026-03-20 16:35:31Epoch 34: 100% 1000/1000 [01:23<00:00, 11.93it/s, a0_loss=0.0147, a0_loss_test=0.00941, diffusion_loss=0.0718, loss=0.0359, loss_test=0.0385, lr=7.36e-5, step=34999]
2026-03-20 16:36:51Epoch 35: 100% 1000/1000 [01:20<00:00, 12.35it/s, a0_loss=0.00997, a0_loss_test=0.0103, diffusion_loss=0.0593, loss=0.0297, loss_test=0.0405, lr=7.22e-5, step=35999]
2026-03-20 16:38:13Epoch 36: 100% 1000/1000 [01:23<00:00, 12.04it/s, a0_loss=0.00317, a0_loss_test=0.0106, diffusion_loss=0.0394, loss=0.0197, loss_test=0.0404, lr=7.08e-5, step=36999]
2026-03-20 16:39:41Epoch 37: 100% 1000/1000 [01:26<00:00, 11.56it/s, a0_loss=0.0165, a0_loss_test=0.0136, diffusion_loss=0.106, loss=0.053, loss_test=0.0464, lr=6.93e-5, step=37999]
2026-03-20 16:41:01Epoch 38: 100% 1000/1000 [01:21<00:00, 12.25it/s, a0_loss=0.00179, a0_loss_test=0.00996, diffusion_loss=0.0426, loss=0.0213, loss_test=0.0389, lr=6.78e-5, step=38999]
2026-03-20 16:42:26Epoch 39: 100% 1000/1000 [01:23<00:00, 11.95it/s, a0_loss=0.011, a0_loss_test=0.00984, diffusion_loss=0.0917, loss=0.0459, loss_test=0.0381, lr=6.64e-5, step=4e+4]
2026-03-20 16:43:50
""".strip()

# ── Parse console ──
def parse_console(text):
    text = re.sub(r"(?=\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})", "\n", text)
    rows = []
    for line in text.splitlines():
        if "Epoch" not in line or "step=" not in line:
            continue
        def grab(pat):
            m = re.search(pat, line)
            return float(m.group(1)) if m else np.nan
        step = grab(r"step=([0-9]+(?:\.[0-9]+)?(?:e\+?[0-9]+)?)")
        if np.isnan(step):
            continue
        step = int(step)
        rows.append((
            step,
            grab(r"(?<!\w)loss=([0-9]+(?:\.[0-9]+)?(?:e[+-]?[0-9]+)?)"),
            grab(r"(?<!\w)loss_test=([0-9]+(?:\.[0-9]+)?(?:e[+-]?[0-9]+)?)"),
            grab(r"(?<!\w)a0_loss=([0-9]+(?:\.[0-9]+)?(?:e[+-]?[0-9]+)?)"),
            grab(r"a0_loss_test=([0-9]+(?:\.[0-9]+)?(?:e[+-]?[0-9]+)?)"),
        ))
    if not rows:
        raise ValueError("No rows parsed.")
    return np.array(sorted(rows, key=lambda x: x[0]), dtype=float)

w = parse_console(WANDB_LOG_TEXT)

# ── Combine ──
def combine(local_arr, extra_steps, extra_vals):
    local_steps = local_arr[:, 0].astype(int)
    local_vals  = local_arr[:, 1].astype(float)
    local_set = set(local_steps.tolist())
    fin = np.isfinite(extra_vals)
    keep = np.array([s not in local_set for s in extra_steps]) & fin
    all_s = np.concatenate([extra_steps[keep], local_steps])
    all_v = np.concatenate([extra_vals[keep],  local_vals])
    order = np.argsort(all_s)
    return np.c_[all_s[order], all_v[order]]

SPECS = [
    ("training_losses",    1),
    ("test_losses",        2),
    ("training_a0_losses", 3),
    ("test_a0_losses",     4),
]

# ── Build output as plain dict of lists (same as pickle.load output) ──
data = {}
for pkl_key, col in SPECS:
    arr = combine(pkl[pkl_key], w[:, 0].astype(int), w[:, col])
    data[pkl_key] = [[int(row[0]), float(row[1])] for row in arr]

data_diff = data
data_diff['_label'] = 'Diffusion (patched)'
data_diff

### Data Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── 4 runs: HP1 FM, HP1 Diffusion, HP2 FM, HP2 Diffusion ──
all_runs = [data_fm_hp, data_fm, data_diff, data_fm_v2]
colors   = ['tab:green', 'tab:blue', 'tab:red', 'tab:orange']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, key, title in [
    (axes[0,0], 'test_losses',       'Test Loss'),
    (axes[0,1], 'test_a0_losses',    'Test a0 Loss'),
    (axes[1,0], 'training_losses',   'Training Loss'),
    (axes[1,1], 'training_a0_losses','Training a0 Loss'),
]:
    for run, c in zip(all_runs, colors):
        arr = np.array(run[key])
        ax.plot(arr[:,0], arr[:,1], label=run['_label'], color=c, alpha=0.8)
    ax.set_xlabel('Step'); ax.set_ylabel('Loss')
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('4-Run Comparison — avoiding-d3il / H8_K20 / seed 6',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('4run_comparison.png', dpi=150)
plt.show()

# ── Summary stats ──
print(f"\n{'='*86}")
print(f"{'Run':<22} {'Final Test Loss':>16} {'Min Test Loss':>16} {'Final a0 Test':>16} {'Min a0 Test':>14}")
print(f"{'='*86}")
for run in all_runs:
    tl = np.array(run['test_losses'])
    a0 = np.array(run['test_a0_losses'])
    print(f"{run['_label']:<22} {tl[-1,1]:>16.6f} {tl[:,1].min():>16.6f} {a0[-1,1]:>16.6f} {a0[:,1].min():>14.6f}")

# ── Convergence speed: step where test loss first drops below threshold ──
print(f"\n{'='*86}")
print("Convergence speed (first step where test_loss < threshold):")
for thresh in [0.20, 0.15, 0.12]:
    print(f"\n  threshold = {thresh}")
    for run, c in zip(all_runs, colors):
        arr = np.array(run['test_losses'])
        below = arr[arr[:,1] < thresh]
        step = int(below[0,0]) if len(below) > 0 else None
        print(f"    {run['_label']:<22}: step {step}" if step else f"    {run['_label']:<22}: never reached")

# ── Overfitting gap (train vs test) at final step ──
print(f"\n{'='*86}")
print("Overfitting gap at final step (train_loss - test_loss):")
for run in all_runs:
    tr = np.array(run['training_losses'])
    te = np.array(run['test_losses'])
    n  = min(len(tr), len(te))
    gap = tr[n-1,1] - te[n-1,1]
    print(f"  {run['_label']:<22}: {gap:+.6f}  ({'overfit' if gap < -0.01 else 'underfit' if gap > 0.01 else 'balanced'})")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

all_runs = [data_fm_v3, data_fm_v2, data_diff]
colors = ['tab:green', 'tab:blue', 'tab:red']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, key, title in [
    (axes[0,0], 'test_losses',       'Test Loss'),
    (axes[0,1], 'test_a0_losses',    'Test a0 Loss'),
    (axes[1,0], 'training_losses',   'Training Loss'),
    (axes[1,1], 'training_a0_losses','Training a0 Loss'),
]:
    for run, c in zip(all_runs, colors):
        arr = np.array(run[key])
        ax.plot(arr[:,0], arr[:,1], label=run['_label'], color=c, alpha=0.8)
    ax.set_xlabel('Step'); ax.set_ylabel('Loss')
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('3-Run Comparison — avoiding-d3il / H8_K20 / seed 6', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('3run_comparison.png', dpi=150)
plt.show()

# ── Summary stats ──
print(f"\n{'='*70}")
print(f"{'Run':<22} {'Final Test Loss':>16} {'Min Test Loss':>16} {'Final a0 Test':>16} {'Min a0 Test':>14}")
print(f"{'='*70}")
for run in all_runs:
    tl = np.array(run['test_losses'])
    a0 = np.array(run['test_a0_losses'])
    print(f"{run['_label']:<22} {tl[-1,1]:>16.6f} {tl[:,1].min():>16.6f} {a0[-1,1]:>16.6f} {a0[:,1].min():>14.6f}")

# ── Convergence speed: step where test loss first drops below threshold ──
print(f"\n{'='*70}")
print("Convergence speed (first step where test_loss < threshold):")
for thresh in [0.20, 0.15, 0.12]:
    print(f"\n  threshold = {thresh}")
    for run, c in zip(all_runs, colors):
        arr = np.array(run['test_losses'])
        below = arr[arr[:,1] < thresh]
        step = int(below[0,0]) if len(below) > 0 else None
        print(f"    {run['_label']:<22}: step {step}" if step else f"    {run['_label']:<22}: never reached")

# ── Overfitting gap (train vs test) at final step ──
print(f"\n{'='*70}")
print("Overfitting gap at final step (train_loss - test_loss):")
for run in all_runs:
    tr = np.array(run['training_losses'])
    te = np.array(run['test_losses'])
    n = min(len(tr), len(te))
    gap = tr[n-1,1] - te[n-1,1]
    print(f"  {run['_label']:<22}: {gap:+.6f}  ({'overfit' if gap < -0.01 else 'underfit' if gap > 0.01 else 'balanced'})")

## Load the Plan .npz logs

In [ ]:
import numpy as np

# Path to your specific file
file_path = ''

# allow_pickle=True is MANDATORY for 'object' dtypes in npz
# Using a context manager ('with') is safer for file I/O
with np.load(file_path, allow_pickle=True) as data:

    # Force NumPy to print the entire array instead of truncating with "..."
    np.set_printoptions(threshold=np.inf, precision=6, suppress=True)

    print(f"{'Key':<25} | {'Shape':<15} | {'Data Type':<10}")
    print("=" * 60)

    for key in data.files:
        array = data[key]

        # Header for each entry
        print(f"{key:<25} | {str(array.shape):<15} | {str(array.dtype):<10}")

        # Logic to handle potential object arrays or nested dicts
        if array.dtype == 'object':
            print("--- Object Content ---")
            print(array.tolist()) # Converting to list often reveals the internal structure
        else:
            print(array)

        print("-" * 60)